In [1]:
pip install streamlit scikit-learn pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 92.0 MB/s eta 0:00:00


##Create File

##Full App Code

In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

st.set_page_config(page_title="Food Code Builder", layout="wide")

st.title("🧠 Food Intelligence Code Builder")
st.write("Interactively design and test nutrition scoring logic")

# -------------------------
# Sample dataset
# -------------------------
data = [
    {"food": "Apple", "calories": 95, "protein": 0.5, "fiber": 4.4, "sugar": 19, "sodium": 2},
    {"food": "Chicken Breast", "calories": 165, "protein": 31, "fiber": 0, "sugar": 0, "sodium": 74},
    {"food": "Protein Bar", "calories": 220, "protein": 20, "fiber": 5, "sugar": 12, "sodium": 200},
    {"food": "Soda", "calories": 150, "protein": 0, "fiber": 0, "sugar": 39, "sodium": 45},
    {"food": "Broccoli", "calories": 55, "protein": 3.7, "fiber": 5.2, "sugar": 1.5, "sodium": 33},
    {"food": "Frozen Pizza", "calories": 300, "protein": 12, "fiber": 2, "sugar": 4, "sodium": 700},
]

df = pd.DataFrame(data)

# -------------------------
# Rule-based scoring
# -------------------------
def rule_score(row):
    score = 0
    score += row["protein"] * 1.5
    score += row["fiber"] * 2
    score -= row["sugar"] * 1.2
    score -= row["sodium"] / 100

    if row["sugar"] > 25:
        score -= 10
    if row["sodium"] > 500:
        score -= 8
    if row["fiber"] > 3 and row["sugar"] < 10:
        score += 5

    return score

df["rule_score"] = df.apply(rule_score, axis=1)

# -------------------------
# ML model
# -------------------------
features = ["calories", "protein", "fiber", "sugar", "sodium"]

df["ml_target"] = (
    df["protein"] * 1.2 +
    df["fiber"] * 2 -
    df["sugar"] * 1.5 -
    df["sodium"] / 120
)

model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(df[features], df["ml_target"])

df["ml_score"] = model.predict(df[features])

# Normalize
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df["rule_norm"] = normalize(df["rule_score"])
df["ml_norm"] = normalize(df["ml_score"])
df["hybrid_score"] = 0.6 * df["rule_norm"] + 0.4 * df["ml_norm"]

# -------------------------
# User Input
# -------------------------
st.sidebar.header("Enter Food Nutrition")

calories = st.sidebar.slider("Calories", 0, 500, 150)
protein = st.sidebar.slider("Protein (g)", 0.0, 50.0, 5.0)
fiber = st.sidebar.slider("Fiber (g)", 0.0, 15.0, 2.0)
sugar = st.sidebar.slider("Sugar (g)", 0.0, 50.0, 10.0)
sodium = st.sidebar.slider("Sodium (mg)", 0, 1000, 100)

input_df = pd.DataFrame([{
    "calories": calories,
    "protein": protein,
    "fiber": fiber,
    "sugar": sugar,
    "sodium": sodium
}])

# -------------------------
# Compute Scores
# -------------------------
input_df["rule_score"] = input_df.apply(rule_score, axis=1)
input_df["ml_score"] = model.predict(input_df[features])

input_df["rule_norm"] = normalize(pd.concat([df["rule_score"], input_df["rule_score"]])).iloc[-1]
input_df["ml_norm"] = normalize(pd.concat([df["ml_score"], input_df["ml_score"]])).iloc[-1]

input_df["hybrid_score"] = 0.6 * input_df["rule_norm"] + 0.4 * input_df["ml_norm"]

# -------------------------
# Display Results
# -------------------------
st.subheader("📊 Your Food Score")

st.metric("Hybrid Score", f"{input_df['hybrid_score'].values[0]:.2f}")

st.write("### Score Breakdown")
st.write(f"Rule Score: {input_df['rule_score'].values[0]:.2f}")
st.write(f"ML Score: {input_df['ml_score'].values[0]:.2f}")

# -------------------------
# Interpretation Layer (VERY IMPORTANT)
# -------------------------
st.write("### 🧠 Interpretation")

if sugar > 25:
    st.warning("High sugar detected → strong negative impact")

if sodium > 500:
    st.warning("High sodium detected → penalty applied")

if fiber > 3 and sugar < 10:
    st.success("High fiber + low sugar → positive health signal")

if protein > 15:
    st.info("High protein contributes positively, but consider sodium balance")

# -------------------------
# Show dataset
# -------------------------
if st.checkbox("Show Sample Food Rankings"):
    st.write(df[["food", "hybrid_score"]].sort_values(by="hybrid_score", ascending=False))

Writing app.py


##Run It

In [3]:
pip install pyngrok -q

In [4]:
from pyngrok import ngrok
import subprocess
import os

# Kill any running ngrok tunnels
ngrok.kill()


# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3BoP8JcJquvFY4E7eOxvTobTLF8_7LfSPy9wwubtuQcpvBV5x")

# Start ngrok tunnel for Streamlit on port 8501
public_url = ngrok.connect(8501)
print(f"Streamlit App URL: {public_url}")

# Run the Streamlit app in the background
# Using subprocess.Popen to run it non-blocking
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "True"])

Streamlit App URL: NgrokTunnel: "https://ruby-calvus-nondomestically.ngrok-free.dev" -> "http://localhost:8501"


<Popen: returncode: None args: ['streamlit', 'run', 'app.py', '--server.port...>

##Code Builder UI

In [5]:
%%writefile app.py
import streamlit as st
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

st.set_page_config(page_title="Food Code Builder UI", layout="wide")

st.title("🧠 Food Intelligence Code Builder UI")
st.write("Interactively design and test nutrition scoring logic")

# -------------------------
# Sample dataset
# -------------------------
data = [
    {"food": "Apple", "calories": 95, "protein": 0.5, "fiber": 4.4, "sugar": 19, "sodium": 2},
    {"food": "Chicken Breast", "calories": 165, "protein": 31, "fiber": 0, "sugar": 0, "sodium": 74},
    {"food": "Protein Bar", "calories": 220, "protein": 20, "fiber": 5, "sugar": 12, "sodium": 200},
    {"food": "Soda", "calories": 150, "protein": 0, "fiber": 0, "sugar": 39, "sodium": 45},
    {"food": "Broccoli", "calories": 55, "protein": 3.7, "fiber": 5.2, "sugar": 1.5, "sodium": 33},
    {"food": "Frozen Pizza", "calories": 300, "protein": 12, "fiber": 2, "sugar": 4, "sodium": 700},
]

df = pd.DataFrame(data)

# -------------------------
# CODE BUILDER PANEL
# -------------------------
st.sidebar.header("⚙️ Build Your Nutrition Code")

st.sidebar.subheader("Weights")
protein_weight = st.sidebar.slider("Protein Weight", 0.0, 3.0, 1.5)
fiber_weight = st.sidebar.slider("Fiber Weight", 0.0, 3.0, 2.0)
sugar_weight = st.sidebar.slider("Sugar Penalty", 0.0, 3.0, 1.2)
sodium_scale = st.sidebar.slider("Sodium Scaling", 50, 300, 100)

st.sidebar.subheader("Threshold Rules")

high_sugar_threshold = st.sidebar.slider("High Sugar Threshold", 10, 50, 25)
high_sugar_penalty = st.sidebar.slider("High Sugar Penalty", 0, 20, 10)

high_sodium_threshold = st.sidebar.slider("High Sodium Threshold", 200, 1000, 500)
high_sodium_penalty = st.sidebar.slider("High Sodium Penalty", 0, 20, 8)

fiber_bonus_toggle = st.sidebar.checkbox("Enable Fiber Bonus", True)
fiber_bonus_value = st.sidebar.slider("Fiber Bonus", 0, 10, 5)

# -------------------------
# RULE ENGINE (DYNAMIC)
# -------------------------
def compute_score(row):
    score = 0

    # Base weights
    score += row["protein"] * protein_weight
    score += row["fiber"] * fiber_weight
    score -= row["sugar"] * sugar_weight
    score -= row["sodium"] / sodium_scale

    # Threshold penalties
    if row["sugar"] > high_sugar_threshold:
        score -= high_sugar_penalty

    if row["sodium"] > high_sodium_threshold:
        score -= high_sodium_penalty

    # Bonus logic
    if fiber_bonus_toggle:
        if row["fiber"] > 3 and row["sugar"] < 10:
            score += fiber_bonus_value

    return score

df["rule_score"] = df.apply(compute_score, axis=1)

# -------------------------
# ML LAYER
# -------------------------
features = ["calories", "protein", "fiber", "sugar", "sodium"]

df["ml_target"] = (
    df["protein"] * 1.2 +
    df["fiber"] * 2 -
    df["sugar"] * 1.5 -
    df["sodium"] / 120
)

model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(df[features], df["ml_target"])
df["ml_score"] = model.predict(df[features])

# Normalize
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df["rule_norm"] = normalize(df["rule_score"])
df["ml_norm"] = normalize(df["ml_score"])

# Hybrid control
st.sidebar.subheader("Hybrid Control")
rule_weight = st.sidebar.slider("Rule Weight", 0.0, 1.0, 0.6)
ml_weight = 1 - rule_weight

df["hybrid_score"] = rule_weight * df["rule_norm"] + ml_weight * df["ml_norm"]

# -------------------------
# MAIN PANEL
# -------------------------
st.subheader("📊 Food Rankings")

df_sorted = df.sort_values(by="hybrid_score", ascending=False)

st.dataframe(df_sorted[["food", "rule_score", "ml_score", "hybrid_score"]])

# -------------------------
# VISUALIZATION
# -------------------------
st.subheader("📈 Score Visualization")

st.bar_chart(df_sorted.set_index("food")["hybrid_score"])

# -------------------------
# LOGIC INSPECTOR (VERY IMPORTANT)
# -------------------------
st.subheader("🔍 Logic Inspector")

selected_food = st.selectbox("Select a food to inspect", df["food"])

row = df[df["food"] == selected_food].iloc[0]

st.write("### Feature Values")
st.write(row[["protein", "fiber", "sugar", "sodium"]])

st.write("### Rule Contributions")

st.write(f"Protein Contribution: {row['protein'] * protein_weight:.2f}")
st.write(f"Fiber Contribution: {row['fiber'] * fiber_weight:.2f}")
st.write(f"Sugar Penalty: {-row['sugar'] * sugar_weight:.2f}")
st.write(f"Sodium Penalty: {-row['sodium'] / sodium_scale:.2f}")

if row["sugar"] > high_sugar_threshold:
    st.warning("High sugar penalty applied")

if row["sodium"] > high_sodium_threshold:
    st.warning("High sodium penalty applied")

if fiber_bonus_toggle and row["fiber"] > 3 and row["sugar"] < 10:
    st.success("Fiber bonus applied")

Overwriting app.py


In [6]:
from pyngrok import ngrok
import subprocess
import os

# Kill any running ngrok tunnels
ngrok.kill()


# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3BoP8JcJquvFY4E7eOxvTobTLF8_7LfSPy9wwubtuQcpvBV5x")

# Start ngrok tunnel for Streamlit on port 8501
public_url = ngrok.connect(8501)
print(f"Streamlit App URL: {public_url}")

# Run the Streamlit app in the background
# Using subprocess.Popen to run it non-blocking
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "True"])

Streamlit App URL: NgrokTunnel: "https://ruby-calvus-nondomestically.ngrok-free.dev" -> "http://localhost:8501"


<Popen: returncode: None args: ['streamlit', 'run', 'app.py', '--server.port...>